## 🔰PyTorchでニューラルネットワーク基礎 #26 【トークナイズ編・Unigram LM】
Unigram Language Modelによるトークナイズ

### 内容
* Qiitaの記事と連動しています

### データについて
cc100から日本語の部分を2万行抽出したもの。text列に対応する言語の文書
* 日本語：tiny_cc100_ja.csv
* わかち書き版：tiny_cc100_ja_wakati.csv

### ちょっとした注意点
* **データやトークナイザーの保存先に注意**。適宜修正してください。
* 利用ライブラリ：tokenizers
* データ数が多い場合はシャッフルする部分を少し変更しないとNG

In [ ]:
import random
import pandas as pd
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers

# (1) Unigram を使う（unk_token は trainer 側で指定するのがポイント）
tokenizer = Tokenizer(models.Unigram())
#tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
#tokenizer = Tokenizer(models.WordPiece(unk_token="<unk>"))


# (2) 正規化つけてみた
tokenizer.normalizer = normalizers.NFKC()

# (3)
# 分かち書きの時ONにして効果を確認してみたいな
#tokenizer.pre_tokenizer = pre_tokenizers.Metaspace(replacement="▁")

#(4) <unk>の指定
trainer = trainers.UnigramTrainer(
    vocab_size=10_000,
    special_tokens=["<pad>", "<bos>", "<eos>", "<unk>", "<mask>"],
    unk_token="<unk>",
    shrinking_factor=0.75,
    max_piece_length=16,
    n_sub_iterations=2,
)

# (5) csvファイルから直接学習
paths = ["./data/tiny_cc100_ja.csv"]

# (6) ランダムにしなくてもいいけど、面倒なので前回の多言語仕様をそのまま使ってしまった
def mixed_iterator(paths):
    texts = []
    for p in paths:
        # text列だけ読み込む
        df = pd.read_csv(p)
        texts.extend(df["text"].tolist())   
    # 一気にシャッフル（数百万件程度までならこの方法でOKなはず）
    random.shuffle(texts)  
    for t in texts:
        yield t


# (7) 学習
tokenizer.train_from_iterator(mixed_iterator(paths), trainer=trainer)

# (8) 保存
tokenizer.save("unigram_10k.json")

### 動作の確認

In [ ]:
from tokenizers import Tokenizer

# 保存したトークナイザーで確認
tokenizer = Tokenizer.from_file("unigram_10k.json")

print("特殊トークンID:")
print(f"<pad>: {tokenizer.token_to_id('<pad>')}")
print(f"<bos>: {tokenizer.token_to_id('<bos>')}")
print(f"<eos>: {tokenizer.token_to_id('<eos>')}")
print(f"<unk>: {tokenizer.token_to_id('<unk>')}")
print(f"<mask>: {tokenizer.token_to_id('<mask>')}")
print(f"size: {tokenizer.get_vocab_size()}")

特殊トークンID:
<pad>: 0
<bos>: 1
<eos>: 2
<unk>: 3
<mask>: 4
size: 10000


In [3]:
text_list = [
    "これは日本語のテストです",
    "Awesome blog! Do you have any suggestions",
    "📷　정교하면서 완벽하게 아날로그 카메라를 재현한 것 같다.",
    "你好",
    "𐀀𐀁"
]
    
for text in text_list:
    encoded = tokenizer.encode(text)
    print(f"文章: {text}")
    print("トークン:", encoded.tokens)
    print("ID:", encoded.ids)
    print(f"デコード: {tokenizer.decode(encoded.ids)}\n")

文章: これは日本語のテストです
トークン: ['これは', '日本語', 'の', 'テスト', 'です']
ID: [391, 1324, 6, 4120, 207]
デコード: これは 日本語 の テスト です

文章: Awesome blog! Do you have any suggestions
トークン: ['A', 'w', 'e', 's', 'o', 'm', 'e', ' ', 'b', 'l', 'o', 'g', '! ', 'D', 'o', ' ', 'y', 'o', 'u', ' ', 'h', 'a', 'v', 'e', ' ', 'an', 'y', ' ', 's', 'u', 'g', 'g', 'est', 'i', 'on', 's']
ID: [121, 1133, 226, 522, 171, 318, 226, 14, 951, 547, 171, 600, 1005, 361, 171, 14, 1309, 171, 415, 14, 647, 276, 1257, 226, 14, 3229, 1309, 14, 522, 415, 600, 600, 6908, 255, 1871, 522]
デコード: A w e s o m e   b l o g !  D o   y o u   h a v e   an y   s u g g est i on s

文章: 📷　정교하면서 완벽하게 아날로그 카메라를 재현한 것 같다.
トークン: ['📷', ' ', '정교하면서', ' ', '완벽하게', ' ', '아날로그', ' ', '카메라를', ' ', '재현한', ' ', '것', ' ', '같다', '.']
ID: [3, 14, 3, 14, 3, 14, 3, 14, 3, 14, 3, 14, 3, 14, 3, 254]
デコード:               .

文章: 你好
トークン: ['你', '好']
ID: [3, 2149]
デコード: 好

文章: 𐀀𐀁
トークン: ['𐀀𐀁']
ID: [3]
デコード: 



### NFKCの効果を確認してみた

In [4]:
nfkc = normalizers.NFKC()

samples = [
    "ＡＢＣ１２３",
    "ﾃｽﾄです",
    "㍿○△□",
    "é",         # 合成済み
]

for s in samples:
    print("raw :", repr(s))
    print("nfkc:", repr(nfkc.normalize_str(s)))
    print()

raw : 'ＡＢＣ１２３'
nfkc: 'ABC123'

raw : 'ﾃｽﾄです'
nfkc: 'テストです'

raw : '㍿○△□'
nfkc: '株式会社○△□'

raw : 'é'
nfkc: 'é'

